<a href="https://colab.research.google.com/github/4iharu/Tugas-4-Sistem-temu-kembali/blob/main/240210502026_Nur_Insyirah_Apriany_R_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Kumpulan 4 Dokumen Pendek (Topik Komputer & IR)
docs = [
    "Sistem komputer modern sangat membutuhkan prosesor yang cepat.",
    "Kecerdasan buatan secara perlahan mengubah cara komputer memproses data.",
    "Prosesor dan memori adalah komponen utama dari sistem komputer.",
    "Sistem temu kembali informasi bertugas mencari data secara efektif."
]

# 1. Preprocessing Sederhana (Case Folding + Cleaning + Tokenisasi + Stopwords)
stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()

def simple_preprocess(text):
    text = text.lower()
    import re
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [w for w in text.split() if w not in stopwords]
    return ' '.join(tokens)

clean_docs = [simple_preprocess(d) for d in docs]
print("Dokumen Bersih:\n", clean_docs, "\n")

# 2. Bag of Words (Raw Count) & 3. Perhitungan Manual
# Ekstrak semua term unik (vocabulary)
vocab = sorted(list(set(' '.join(clean_docs).split())))
N = len(clean_docs)

# Hitung TF (Term Frequency)
tf_data = []
for doc in clean_docs:
    tokens = doc.split()
    tf_dict = {term: tokens.count(term) for term in vocab}
    tf_data.append(tf_dict)

df_tf = pd.DataFrame(tf_data, index=[f"Doc {i+1}" for i in range(N)])

# Hitung DF (Document Frequency) dan IDF (Inverse Document Frequency)
# Standar IDF scikit-learn smooth = ln((1+N)/(1+DF)) + 1
df_term = (df_tf > 0).sum(axis=0)
idf_term = np.log((1 + N) / (1 + df_term)) + 1

# Hitung TF-IDF Manual (TF * IDF) lalu di-normalize (L2 norm) agar persis scikit-learn
tfidf_manual = df_tf * idf_term
tfidf_manual_norm = tfidf_manual.div(np.linalg.norm(tfidf_manual, axis=1), axis=0)

print("--- MATRIKS TF-IDF MANUAL ---")
display(tfidf_manual_norm)

# 4 & 5. Implementasi Scikit-Learn dan Perbandingan
vectorizer = TfidfVectorizer(vocabulary=vocab)
tfidf_sklearn = vectorizer.fit_transform(clean_docs)
df_tfidf_sklearn = pd.DataFrame(tfidf_sklearn.toarray(), columns=vocab, index=[f"Doc {i+1}" for i in range(N)])

print("\n--- MATRIKS TF-IDF SCIKIT-LEARN ---")
display(df_tfidf_sklearn)

# Verifikasi kesamaan (selisih kecil akibat pembulatan *floating point*)
print("\nPerbedaan Maksimal Manual vs Sklearn:", (tfidf_manual_norm - df_tfidf_sklearn).abs().max().max())

Dokumen Bersih:
 ['sistem komputer modern sangat membutuhkan prosesor cepat', 'kecerdasan buatan perlahan mengubah cara komputer memproses data', 'prosesor memori komponen utama sistem komputer', 'sistem temu informasi bertugas mencari data efektif'] 

--- MATRIKS TF-IDF MANUAL ---


,bertugas,buatan,cara,cepat,data,efektif,informasi,kecerdasan,komponen,komputer,...,memproses,mencari,mengubah,modern,perlahan,prosesor,sangat,sistem,temu,utama
Doc 1,0.000000,0.000000,0.000000,0.428888,0.000000,0.000000,0.000000,0.000000,0.000000,0.273754,...,0.000000,0.000000,0.000000,0.428888,0.000000,0.338140,0.428888,0.273754,0.000000,0.000000
Doc 2,0.000000,0.377184,0.377184,0.000000,0.297376,0.000000,0.000000,0.377184,0.000000,0.240752,...,0.377184,0.000000,0.377184,0.000000,0.377184,0.000000,0.000000,0.000000,0.000000,0.000000
Doc 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.474771,0.303040,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.374315,0.000000,0.303040,0.000000,0.474771
Doc 4,0.407265,0.000000,0.000000,0.000000,0.321093,0.407265,0.407265,0.000000,0.000000,0.000000,...,0.000000,0.407265,0.000000,0.000000,0.000000,0.000000,0.000000,0.259952,0.407265,0.000000



--- MATRIKS TF-IDF SCIKIT-LEARN ---


,bertugas,buatan,cara,cepat,data,efektif,informasi,kecerdasan,komponen,komputer,...,memproses,mencari,mengubah,modern,perlahan,prosesor,sangat,sistem,temu,utama
Doc 1,0.000000,0.000000,0.000000,0.428888,0.000000,0.000000,0.000000,0.000000,0.000000,0.273754,...,0.000000,0.000000,0.000000,0.428888,0.000000,0.338140,0.428888,0.273754,0.000000,0.000000
Doc 2,0.000000,0.377184,0.377184,0.000000,0.297376,0.000000,0.000000,0.377184,0.000000,0.240752,...,0.377184,0.000000,0.377184,0.000000,0.377184,0.000000,0.000000,0.000000,0.000000,0.000000
Doc 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.474771,0.303040,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.374315,0.000000,0.303040,0.000000,0.474771
Doc 4,0.407265,0.000000,0.000000,0.000000,0.321093,0.407265,0.407265,0.000000,0.000000,0.000000,...,0.000000,0.407265,0.000000,0.000000,0.000000,0.000000,0.000000,0.259952,0.407265,0.000000



Perbedaan Maksimal Manual vs Sklearn: 0.0


Analisis Pembobotan TF-IDF:
Term yang memiliki bobot TF-IDF tertinggi pada masing-masing dokumen adalah kata-kata yang spesifik dan dominan pada dokumen tersebut namun sangat jarang ditemukan di dokumen lain dalam korpus. Contohnya, kata "cerdas", "ubah", dan "proses" akan mendapat bobot tertinggi di Dokumen 2 karena kata tersebut hanya muncul di sana (IDF maksimal). Term-term eksklusif ini sangat penting karena berfungsi sebagai kata kunci pembeda (discriminative keywords) yang merepresentasikan identitas unik sebuah dokumen, sehingga ketika user memasukkan query yang spesifik, sistem dapat langsung memberikan dokumen yang paling relevan berdasarkan bobot tersebut.